In [ ]:
"""
╔══════════════════════════════════════════════════════════════╗
║    LAB SESSION 2 — PHASE 3 : MODÈLES DEEP LEARNING          ║
║                   models_lab.py                              ║
║                                                              ║
║  Modèle 1 : LSTM simple (baseline)                           ║
║  Modèle 2 : BiGRU + Multi-Head Attention (hybride avancé)    ║
║                                                              ║
║  Tâche    : Régression — prédire Close(t+1)                  ║
║  Métrique : RMSE (Root Mean Squared Error)                   ║
╚══════════════════════════════════════════════════════════════╝
"""

import logging
import numpy as np
import pandas as pd
import joblib
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from datetime import datetime, timedelta

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

# ─────────────────────────────────────────────
# Dossiers Google Colab
# ─────────────────────────────────────────────
BASE_DIR    = Path("/content")

MODELS_DIR  = BASE_DIR / "models"

REPORTS_DIR = BASE_DIR / "reports"

FIGURES_DIR = REPORTS_DIR / "figures"

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s  [%(levelname)s]  %(message)s")
logger = logging.getLogger(__name__)
N_FEATURES = 7

# ─────────────────────────────────────────────
#  MODÈLE 1 — LSTM SIMPLE (Baseline)
# ─────────────────────────────────────────────

def build_lstm_simple(window: int = 60, n_features: int = N_FEATURES, units: int = 50) -> Model:
    """
    LSTM simple — réseau de base pour la prévision de séries temporelles.

    Architecture :
      LSTM(50) → Dropout(0.2) → LSTM(50) → Dropout(0.2) → Dense(25) → Dense(1)

    C'est l'architecture classique utilisée dans la littérature Forex.
    """
    inp = Input(shape=(window, n_features), name="input")

    x = layers.LSTM(units, return_sequences=True, name="lstm_1")(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.LSTM(units, return_sequences=False, name="lstm_2")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(25, activation="relu")(x)
    out = layers.Dense(1, name="output")(x)    # Régression → pas de sigmoid

    model = Model(inputs=inp, outputs=out, name="LSTM_Simple")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mean_squared_error",
        metrics=["mae"],
    )
    logger.info(f"🔵 LSTM Simple — {model.count_params():,} paramètres")
    return model


# ─────────────────────────────────────────────
#  MODÈLE 2 — BiGRU + Multi-Head Attention
# ─────────────────────────────────────────────

def build_bigru_attention(
    window:    int = 60,
    units:     int = 64,
    num_heads: int = 4,
    dropout:   float = 0.2,
) -> Model:
    """
    Modèle hybride avancé : BiGRU + Multi-Head Attention.

    Principe :
      BiGRU  : lit la séquence dans les 2 sens → capture tendances haussières ET baissières
      MHA    : pondère les timesteps les plus informatifs (mécanisme d'attention)

    Architecture :
      BiGRU(64) → LayerNorm → MultiHeadAttention(4 heads) → BiGRU(32) → Dense → Dense(1)
    """
    inp = Input(shape=(window, N_FEATURES), name="input")

    # ── Bloc BiGRU 1 ─────────────────────────────────────────────
    x = layers.Bidirectional(
        layers.GRU(units, return_sequences=True), name="bigru_1"
    )(inp)
    x = layers.LayerNormalization()(x)
    x = layers.Dropout(dropout)(x)

    # ── Multi-Head Attention ──────────────────────────────────────
    # key_dim = units * 2 // num_heads (BiGRU double les unités)
    key_dim = (units * 2) // num_heads
    attn_out, attn_weights = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim,
        name="multi_head_attention",
    )(x, x, return_attention_scores=True)

    # Connexion résiduelle (standard dans les Transformers)
    x = layers.Add()([x, attn_out])
    x = layers.LayerNormalization()(x)
    x = layers.Dropout(dropout)(x)

    # ── Bloc BiGRU 2 ─────────────────────────────────────────────
    x = layers.Bidirectional(
        layers.GRU(units // 2, return_sequences=False), name="bigru_2"
    )(x)
    x = layers.LayerNormalization()(x)
    x = layers.Dropout(dropout)(x)

    # ── Tête de régression ────────────────────────────────────────
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(dropout / 2)(x)
    out = layers.Dense(1, name="output")(x)

    model = Model(inputs=inp, outputs=out, name="BiGRU_MultiHead_Attention")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=5e-4, clipnorm=1.0),
        loss="mean_squared_error",
        metrics=["mae"],
    )
    logger.info(f"🔴 BiGRU + MHA — {model.count_params():,} paramètres")
    return model


# ─────────────────────────────────────────────
#  ENTRAÎNEMENT
# ─────────────────────────────────────────────

def train_model(
    model:      Model,
    X_train:    np.ndarray,
    y_train:    np.ndarray,
    X_val:      np.ndarray,
    y_val:      np.ndarray,
    pair:       str,
    model_name: str,
    epochs:     int = 100,
    batch_size: int = 32,
) -> keras.callbacks.History:
    """Entraîne le modèle avec EarlyStopping et sauvegarde le meilleur."""
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    best_path = MODELS_DIR / f"{pair}_{model_name}_best_lab.keras"

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=15,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                          patience=7, min_lr=1e-6, verbose=1),
        ModelCheckpoint(str(best_path), monitor="val_loss",
                        save_best_only=True, verbose=0),
    ]

    logger.info(f"🚀 Entraînement {model_name} sur {pair} ...")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        shuffle=False,   # séries temporelles → PAS de shuffle
        verbose=1,
    )

    # Sauvegarde finale
    model.save(str(MODELS_DIR / f"{pair}_{model_name}_final_lab.keras"))
    logger.info(f"   💾 Modèle sauvegardé.")
    return history


# ─────────────────────────────────────────────
#  ÉVALUATION — RMSE
# ─────────────────────────────────────────────

def compute_rmse(
    model:   Model,
    X_test:  np.ndarray,
    y_test:  np.ndarray,
    scaler,
    pair:    str,
    model_name: str,
) -> dict:
    """
    Calcule le RMSE en valeur réelle (non normalisée).

    RMSE = sqrt( mean( (y_pred - y_true)² ) )
    """


    # Prédictions normalisées
    y_pred_scaled = model.predict(X_test, verbose=0)

    # Dénormalisation → valeurs réelles
    dummy_pred = np.zeros((len(y_pred_scaled), 7))
    dummy_true = np.zeros((len(y_test), 7))

    dummy_pred[:, 0] = y_pred_scaled.flatten()
    dummy_true[:, 0] = y_test.flatten()

    y_pred_real = scaler.inverse_transform(dummy_pred)[:, 0]
    y_true_real = scaler.inverse_transform(dummy_true)[:, 0]

    rmse = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
    mae  = np.mean(np.abs(y_pred_real - y_true_real))
    mape = np.mean(np.abs((y_pred_real - y_true_real) / y_true_real)) * 100

    logger.info(f"   📊 {model_name} | {pair} → RMSE={rmse:.6f} | MAE={mae:.6f} | MAPE={mape:.4f}%")
    r2 = r2_score(y_true_real, y_pred_real)

    direction_true = np.sign(np.diff(y_true_real.flatten()))
    direction_pred = np.sign(np.diff(y_pred_real.flatten()))

    direction_acc = np.mean(
        direction_true == direction_pred
    ) * 100

    return {
        "pair":       pair,
        "model":      model_name,
        "rmse":       round(float(rmse), 6),
        "mae":        round(float(mae),  6),
        "mape":       round(float(mape), 4),
        "y_pred": y_pred_real,
        "y_true": y_true_real,
        "r2": round(float(r2), 4),
        "direction_acc": round(float(direction_acc), 2),
    }


# ─────────────────────────────────────────────
#  PRÉDICTIONS FUTURES 2026
# ─────────────────────────────────────────────

def predict_future_2026_dynamique(
    model,
    df_raw_base: pd.DataFrame,  # Le DataFrame initial non standardisé (les 60 derniers jours de 2025)
    scaler,
    window: int = 60,
    n_days: int = 252,
    last_date: str = "2025-12-31"
):
    """
    Génère des prédictions futures pour 2026 en recalculant REELLEMENT 
    les indicateurs techniques à chaque pas pour éviter le biais mathématique.
    """
    from utils.data_loader import compute_technical_indicators
    
    # 1. On prépare un DataFrame de travail contenant les 'window' derniers jours de 2025
    # Colonnes requises : Date, Close (et les autres si existantes, elles seront écrasées)
    working_df = df_raw_base.copy().tail(window).reset_index(drop=True)
    
    features_order = ["Close", "SMA_10", "EMA_10", "RSI", "MACD", "Returns", "Volatility"]
    predictions_real = []
    future_dates = []
    
    last_dt = pd.to_datetime(last_date)
    
    for i in range(n_days):
        # a. Calculer les indicateurs sur les données brutes glissantes actuelles
        df_indicators = compute_technical_indicators(working_df)
        
        # b. Extraire la fenêtre de taille 'window' et l'ordonner selon l'entraînement
        current_window_raw = df_indicators[features_order].tail(window).values
        
        # c. Normaliser la fenêtre avec le scaler [0, 1]
        current_window_scaled = scaler.transform(current_window_raw)
        
        # d. Reshape pour Keras (1, window, 7) et Inférence
        X_input = current_window_scaled.reshape(1, window, len(features_order))
        pred_scaled = model.predict(X_input, verbose=0)[0, 0]
        
        # e. Dénormaliser la prédiction Close pour l'ajouter au DataFrame brut
        # On utilise une matrice fantôme (dummy) pour l'inverse_transform
        dummy = np.zeros((1, len(features_order)))
        dummy[0, 0] = pred_scaled
        pred_close_real = scaler.inverse_transform(dummy)[0, 0]
        
        predictions_real.append(pred_close_real)
        
        # f. Avancer d'un jour ouvré
        next_date = last_dt + timedelta(days=1)
        while next_date.weekday() >= 5:  # Skip weekends
            next_date += timedelta(days=1)
        last_dt = next_date
        future_dates.append(next_date.strftime("%Y-%m-%d"))
        
        # g. MISE À JOUR DU DATAFRAME : On crée la nouvelle ligne brute
        new_row = pd.DataFrame({
            "Date": [next_date],
            "Close": [pred_close_real]
        })
        
        # On l'ajoute au tableau et on ne garde que les 'window' derniers pas pour le prochain tour
        working_df = pd.concat([working_df, new_row], ignore_index=True).tail(window).reset_index(drop=True)
        
    return np.array(predictions_real), future_dates


# ─────────────────────────────────────────────
#  VISUALISATIONS
# ─────────────────────────────────────────────

def plot_training_curves(history, pair: str, model_name: str) -> None:
    """Courbes de perte train/validation."""
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    h = history.history

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"{model_name} — {pair} — Courbes d'Entraînement",
                 fontsize=13, fontweight="bold")

    epochs = range(1, len(h["loss"]) + 1)

    ax1.plot(epochs, h["loss"],     label="Train Loss", color="steelblue", lw=2)
    ax1.plot(epochs, h["val_loss"], label="Val Loss",   color="darkorange", lw=2, ls="--")
    ax1.set_title("MSE Loss")
    ax1.set_xlabel("Epoch"); ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, h["mae"],     label="Train MAE", color="seagreen", lw=2)
    ax2.plot(epochs, h["val_mae"], label="Val MAE",   color="tomato",   lw=2, ls="--")
    ax2.set_title("MAE")
    ax2.set_xlabel("Epoch"); ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    path = FIGURES_DIR / f"{pair}_{model_name}_training.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    logger.info(f"   📊 {path.name}")


def plot_predictions_vs_real(
    results:     dict,
    dates_test:  np.ndarray,
    pair:        str,
    model_name:  str,
) -> None:
    """Graphique : prédictions vs valeurs réelles sur le test set."""
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(14, 5))

    dates = pd.to_datetime(dates_test)
    ax.plot(dates, results["y_true"], label="Valeurs Réelles",
            color="steelblue", lw=1.5)
    ax.plot(dates, results["y_pred"], label=f"Prédictions {model_name}",
            color="darkorange", lw=1.5, alpha=0.85)

    ax.set_title(
        f"{model_name} — {pair} — Test Set\n"
        f"RMSE = {results['rmse']:.6f} | MAE = {results['mae']:.6f} | MAPE = {results['mape']:.2f}%",
        fontsize=12, fontweight="bold"
    )
    ax.set_xlabel("Date"); ax.set_ylabel("Taux de change")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.xticks(rotation=30)
    plt.tight_layout()

    path = FIGURES_DIR / f"{pair}_{model_name}_predictions.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    logger.info(f"   📊 {path.name}")


def plot_future_2026(
    close_raw:    np.ndarray,
    dates_all:    np.ndarray,
    future_real:  np.ndarray,
    future_dates: list,
    pair:         str,
    model_name:   str,
) -> None:
    """Graphique : historique + prédictions futures 2026."""
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(16, 6))

    # Historique complet
    hist_dates = pd.to_datetime(dates_all)
    ax.plot(hist_dates, close_raw.flatten(),
            color="steelblue", lw=1.2, label="Historique 2015-2025")

    # Zone de prédiction 2026
    fut_dates = pd.to_datetime(future_dates)
    ax.plot(fut_dates, future_real,
            color="darkorange", lw=2, label=f"Prédictions 2026 ({model_name})")

    # Bande de confiance (±5% — approximation visuelle)
    ci = future_real * 0.05
    ax.fill_between(fut_dates,
                    future_real - ci,
                    future_real + ci,
                    alpha=0.2, color="darkorange", label="Intervalle ±5%")

    # Ligne verticale séparant historique et prédictions
    ax.axvline(pd.to_datetime("2026-01-01"), color="red",
               ls="--", lw=1.5, label="Début 2026")

    ax.set_title(
        f"Prédictions Futures 2026 — {pair} ({model_name})\n"
        f"Dernier historique : {hist_dates[-1].date()} | "
        f"Fin prédiction : {fut_dates[-1].date()}",
        fontsize=12, fontweight="bold"
    )
    ax.set_xlabel("Date"); ax.set_ylabel(f"Taux {pair}")
    ax.legend(loc="upper left"); ax.grid(True, alpha=0.3)
    plt.xticks(rotation=30)
    plt.tight_layout()

    path = FIGURES_DIR / f"{pair}_{model_name}_future_2026.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    logger.info(f"   📊 {path.name}")


def plot_comparison_rmse(all_results: list) -> None:
    """Graphique de comparaison RMSE : LSTM vs BiGRU+MHA pour les 3 paires."""
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

    pairs  = ["EURUSD", "EURMAD", "USDMAD"]
    models = ["LSTM_Simple", "BiGRU_MHA"]
    colors = ["steelblue", "darkorange"]

    rmse_data = {m: [] for m in models}
    for pair in pairs:
        for model_name in models:
            r = next((x for x in all_results
                      if x["pair"] == pair and x["model"] == model_name), None)
            rmse_data[model_name].append(r["rmse"] if r else 0)

    x     = np.arange(len(pairs))
    width = 0.35

    fig, ax = plt.subplots(figsize=(11, 6))
    for i, (model_name, color) in enumerate(zip(models, colors)):
        bars = ax.bar(x + i * width, rmse_data[model_name],
                      width, label=model_name, color=color, alpha=0.85)
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.00001,
                    f"{bar.get_height():.5f}",
                    ha="center", va="bottom", fontsize=9)

    ax.set_title("Comparaison RMSE — LSTM Simple vs BiGRU + Multi-Head Attention",
                 fontsize=13, fontweight="bold")
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(pairs, fontsize=12)
    ax.set_ylabel("RMSE (valeur réelle)")
    ax.legend(fontsize=11); ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()

    path = FIGURES_DIR / "comparison_rmse_all_pairs.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    logger.info(f"   📊 {path.name}")


def plot_all_future_2026(futures: dict) -> None:
    """Graphique 3×2 : prédictions 2026 pour les 3 paires × 2 modèles."""
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    pairs  = ["EURUSD", "EURMAD", "USDMAD"]
    models = ["LSTM_Simple", "BiGRU_MHA"]
    colors = ["steelblue", "darkorange"]

    fig, axes = plt.subplots(3, 2, figsize=(16, 14))
    fig.suptitle("Prédictions Futures 2026 — EUR/USD · EUR/MAD · USD/MAD",
                 fontsize=15, fontweight="bold")

    for row, pair in enumerate(pairs):
        for col, (model_name, color) in enumerate(zip(models, colors)):
            ax = axes[row][col]
            key = f"{pair}_{model_name}"
            if key not in futures:
                ax.axis("off"); continue

            future_real, future_dates, close_raw, dates_all = futures[key]

            hist_dates = pd.to_datetime(dates_all[-120:])   # 120 derniers jours
            ax.plot(hist_dates, close_raw.flatten()[-120:],
                    color="gray", lw=1.2, label="Historique (4 mois)")

            fut_dates = pd.to_datetime(future_dates)
            ax.plot(fut_dates, future_real, color=color, lw=2,
                    label=f"{model_name}")
            ci = future_real * 0.05
            ax.fill_between(fut_dates, future_real - ci, future_real + ci,
                            alpha=0.2, color=color)
            ax.axvline(pd.to_datetime("2026-01-01"), color="red", ls="--", lw=1)

            ax.set_title(f"{pair} — {model_name}", fontweight="bold")
            ax.set_ylabel("Taux"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, fontsize=7)

    plt.tight_layout()
    path = FIGURES_DIR / "all_futures_2026.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    logger.info(f"   📊 {path.name}")


# ─────────────────────────────────────────────
#  RAPPORT FINAL
# ─────────────────────────────────────────────

def save_report(all_results: list, futures: dict) -> None:
    """Sauvegarde le rapport complet en JSON avec les bons noms de modèles."""
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)

    # Tableau comparatif
    summary = []
    pairs   = ["EURUSD", "EURMAD", "USDMAD"]
    
    # CORRECTION : On utilise "BiGRU_MHA" pour correspondre exactement aux résultats
    models = ["LSTM_Simple", "BiGRU_MHA"] 

    for pair in pairs:
        row = {"pair": pair}
        for model_name in models:
            r = next((x for x in all_results
                      if x["pair"] == pair and x["model"] == model_name), None)
            if r:
                row[f"{model_name}_rmse"] = r["rmse"]
                row[f"{model_name}_mae"]  = r["mae"]
                row[f"{model_name}_mape"] = r["mape"]
        summary.append(row)

    # Interprétation tendances 2026
    interpretations = {}
    for key, (future_real, future_dates, _, _) in futures.items():
        trend   = "haussière 📈" if future_real[-1] > future_real[0] else "baissière 📉"
        change  = (future_real[-1] - future_real[0]) / future_real[0] * 100
        interpretations[key] = {
            "trend":          trend,
            "pct_change":     round(float(change), 2),
            "start_price":    round(float(future_real[0]),  5),
            "end_price":      round(float(future_real[-1]), 5),
            "period_start":   future_dates[0],
            "period_end":     future_dates[-1],
        }

    report = {
        "lab":       "Lab Session 2 — Forex Prediction (Regression)",
        "date":      datetime.now().isoformat(),
        "pairs":     pairs,
        "models":    models,
        "window":    60,
        "metric":    "RMSE (valeur réelle)",
        "summary":   summary,
        "interpretations_2026": interpretations,
    }

    path = REPORTS_DIR / "lab2_report.json"
    with open(path, "w") as f:
        json.dump(report, f, indent=2)
        
    logger.info(f"📋 Rapport corrigé sauvegardé : {path}")

In [ ]:
import os

# Réduction logs TensorFlow
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

# ─────────────────────────────────────────────
# Chargement des données preprocessées
# ─────────────────────────────────────────────
PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"

PAIRS = [
    "EURUSD",
    "EURMAD",
    "USDMAD"
]

all_results = []

futures = {}

# ─────────────────────────────────────────────
# Boucle principale
# ─────────────────────────────────────────────
for pair in PAIRS:

    print("\n" + "=" * 70)

    print(f"PAIR : {pair}")

    print("=" * 70)

    # ─────────────────────────────────────────
    # Chargement datasets
    # ─────────────────────────────────────────
    X_train = np.load(
        PROCESSED_DATA_DIR / f"{pair}_X_train.npy"
    )

    X_test = np.load(
        PROCESSED_DATA_DIR / f"{pair}_X_test.npy"
    )

    y_train = np.load(
        PROCESSED_DATA_DIR / f"{pair}_y_train.npy"
    )

    y_test = np.load(
        PROCESSED_DATA_DIR / f"{pair}_y_test.npy"
    )

    scaler = joblib.load(
        MODELS_DIR / f"{pair}_scaler.pkl"
    )

    # ─────────────────────────────────────────
    # Split validation
    # ─────────────────────────────────────────
    # Chargement validation
    X_val = np.load(
        PROCESSED_DATA_DIR / f"{pair}_X_val.npy"
    )

    y_val = np.load(
        PROCESSED_DATA_DIR / f"{pair}_y_val.npy"
    )

    X_train_final = X_train
    y_train_final = y_train

    # ─────────────────────────────────────────
    # Backtesting Trading Strategy
    # ─────────────────────────────────────────
    def backtest_strategy(y_true, y_pred):

        signals = []

        profits = []

        for i in range(len(y_pred) - 1):

            current_price = y_true[i]

            next_real = y_true[i + 1]

            prediction = y_pred[i]

            # Signal
            if prediction > current_price:
                signal = 1   # BUY
            else:
                signal = -1  # SELL

            signals.append(signal)

            # Profit
            market_return = (
                next_real - current_price
            ) / current_price

            strategy_return = signal * market_return

            profits.append(strategy_return)

        profits = np.array(profits)

        # Metrics
        total_return = profits.sum() * 100

        win_rate = (
            np.mean(profits > 0) * 100
        )

        sharpe_ratio = (
            profits.mean() /
            (profits.std() + 1e-8)
        ) * np.sqrt(252)

        return {
            "total_return": round(float(total_return), 2),
            "win_rate": round(float(win_rate), 2),
            "sharpe_ratio": round(float(sharpe_ratio), 4),
        }

    # ─────────────────────────────────────────
    # Modèle 1 — LSTM
    # ─────────────────────────────────────────
    lstm_model = build_lstm_simple(window=60)
    lstm_model.summary()

    lstm_history = train_model(
        model=lstm_model,
        X_train=X_train_final,
        y_train=y_train_final,
        X_val=X_val,
        y_val=y_val,
        pair=pair,
        model_name="LSTM_Simple",
        epochs=40,
        batch_size=32,
    )

    lstm_results = compute_rmse(
        model=lstm_model,
        X_test=X_test,
        y_test=y_test,
        scaler=scaler,
        pair=pair,
        model_name="LSTM_Simple",
    )

    all_results.append(lstm_results)
    bt_lstm = backtest_strategy(
        lstm_results["y_true"],
        lstm_results["y_pred"]
    )

    print("\nLSTM BACKTEST")
    print(bt_lstm)

    # ─────────────────────────────────────────
    # Visualisations LSTM
    # ─────────────────────────────────────────
    plot_training_curves(
        lstm_history,
        pair,
        "LSTM_Simple"
    )

    # ─────────────────────────────────────────
    # Modèle 2 — BiGRU + Attention
    # ─────────────────────────────────────────
    bigru_model = build_bigru_attention(window=60)
    bigru_model.summary()

    bigru_history = train_model(
        model=bigru_model,
        X_train=X_train_final,
        y_train=y_train_final,
        X_val=X_val,
        y_val=y_val,
        pair=pair,
        model_name="BiGRU_MHA",
        epochs=40,
        batch_size=32,
    )

    bigru_results = compute_rmse(
        model=bigru_model,
        X_test=X_test,
        y_test=y_test,
        scaler=scaler,
        pair=pair,
        model_name="BiGRU_MHA",
    )

    all_results.append(bigru_results)
    # Backtesting
    bt_results = backtest_strategy(
        bigru_results["y_true"],
        bigru_results["y_pred"]
    )

    print("\nBACKTEST RESULTS")
    print(bt_results)

    # ─────────────────────────────────────────
    # Visualisations BiGRU
    # ─────────────────────────────────────────
    plot_training_curves(
        bigru_history,
        pair,
        "BiGRU_MHA"
    )

    # ─────────────────────────────────────────
    # Comparaison prédictions
    # ─────────────────────────────────────────
    dummy_dates = np.arange(len(y_test))

    plot_predictions_vs_real(
        lstm_results,
        dummy_dates,
        pair,
        "LSTM_Simple"
    )

    plot_predictions_vs_real(
        bigru_results,
        dummy_dates,
        pair,
        "BiGRU_MHA"
    )

    # ─────────────────────────────────────────
    # Choix meilleur modèle
    # ─────────────────────────────────────────
    best_model = (
        bigru_model
        if bigru_results["rmse"] < lstm_results["rmse"]
        else lstm_model
    )

    best_name = (
        "BiGRU_MHA"
        if bigru_results["rmse"] < lstm_results["rmse"]
        else "LSTM_Simple"
    )

    # ─────────────────────────────────────────
    # Rechargement série complète pour plotting
    # ─────────────────────────────────────────
    df_full_hist = pd.read_csv(
        BASE_DIR / "data" / "raw" / f"{pair}.csv",
        parse_dates=["Date"]
    )

    current_pair_close_raw = df_full_hist["Close"].values.reshape(-1, 1)
    current_pair_dates_all = df_full_hist["Date"].values

    features_for_scaling = [
        "Close", "SMA_10", "EMA_10", "RSI", "MACD", "Returns", "Volatility"
    ]
    data_values_for_scaling = df_full_hist[features_for_scaling].values
    scaled_data = scaler.transform(data_values_for_scaling)

    # ─────────────────────────────────────────
    # Prédictions 2026
    # ─────────────────────────────────────────
    future_real, future_dates = predict_future_2026(
        model=best_model,
        scaled_data=scaled_data,
        scaler=scaler,
        window=60,
        n_days=252,
        last_date="2025-12-31",
    )

    futures[f"{pair}_{best_name}"] = (
        future_real,
        future_dates,
        current_pair_close_raw,
        current_pair_dates_all,
    )

    # ─────────────────────────────────────────
    # Visualisation future
    # ─────────────────────────────────────────
    plot_future_2026(
        close_raw=current_pair_close_raw,
        dates_all=current_pair_dates_all,
        future_real=future_real,
        future_dates=future_dates,
        pair=pair,
        model_name=best_name,
    )
    # ─────────────────────────────────────────────────────────────────
    # 💾 SAUVEGARDE & TÉLÉCHARGEMENT AUTOMATIQUE (À METTRE DANS LA BOUCLE)
    # ─────────────────────────────────────────────────────────────────
    print(f"\n💾 Enregistrement et exportation des artefacts pour {pair}...")
    from google.colab import files
    import joblib
    import json

    # 1. Sauvegarde des deux modèles avec leurs vrais noms de variables
    lstm_model.save(f"{pair}_LSTM_Simple.keras")
    bigru_model.save(f"{pair}_BiGRU_MHA.keras")

    # 2. Sauvegarde du scaler avec joblib (comme il a été chargé dans ton code)
    joblib.dump(scaler, f"{pair}_scaler.pkl")

    # 3. Sauvegarde des historiques d'entraînement (.json)
    # Extraction du dictionnaire d'historique s'il s'agit d'un objet de callback Keras
    lstm_dict = lstm_history.history if hasattr(lstm_history, 'history') else lstm_history
    bigru_dict = bigru_history.history if hasattr(bigru_history, 'history') else bigru_history

    with open(f"{pair}_LSTM_Simple_history.json", "w") as f:
        json.dump(lstm_dict, f)
        
    with open(f"{pair}_BiGRU_MHA_history.json", "w") as f:
        json.dump(bigru_dict, f)

    print(f"📥 Envoi des fichiers de {pair} vers le navigateur...")
    # Lancement du téléchargement automatique des 5 fichiers de la paire active
    files.download(f"{pair}_LSTM_Simple.keras")
    files.download(f"{pair}_BiGRU_MHA.keras")
    files.download(f"{pair}_scaler.pkl")
    files.download(f"{pair}_LSTM_Simple_history.json")
    files.download(f"{pair}_BiGRU_MHA_history.json")

# ─────────────────────────────────────────────
# Comparaisons globales
# ─────────────────────────────────────────────
plot_comparison_rmse(all_results)

plot_all_future_2026(futures)

# ─────────────────────────────────────────────
# Rapport final
# ─────────────────────────────────────────────
save_report(all_results, futures)

print("\n✅ LAB SESSION 2 TERMINÉ")